# 00 - Configuración de Unity Catalog

**Objetivo.** Dejar creados los activos base de OceanWatch Analytics en Unity Catalog: catálogo, esquemas y Volume.

**Entradas.** Un identificador corto del equipo en `TEAM_ID` y permisos para crear catálogo, esquemas y Volume.

**Salidas.** `oceanwatch_<equipo>`, los esquemas `landing`, `reference`, `analytics` y el Volume `landing.raw_ais`.

**Prerrequisitos.** Ejecutar en un workspace Databricks con Unity Catalog y serverless habilitados; no requiere secretos ni datos locales.

**Validación.** La última celda ejecuta `SHOW CATALOGS`, `SHOW SCHEMAS` y `DESCRIBE VOLUME`.

In [0]:
# Identificador provisional neutral; reemplazar únicamente tras acordarlo con el equipo.
TEAM_ID = "g06"

# Validación defensiva para usar el identificador de forma segura en sentencias SQL.
if not TEAM_ID.replace("_", "").isalnum() or TEAM_ID != TEAM_ID.lower():
    raise ValueError("TEAM_ID debe usar solo minúsculas, números y guiones bajos")

CATALOG = f"oceanwatch_{TEAM_ID}"
SCHEMA_COMMENTS = {
    "landing": "Datos AIS descargados y descomprimidos desde la fuente oficial NOAA.",
    "reference": "Datos de referencia: World Port Index y catálogo de tipos AIS.",
    "analytics": "Tablas optimizadas y resultados analíticos de OceanWatch.",
}
VOLUME_NAME = "raw_ais"
VOLUME_FQN = f"{CATALOG}.landing.{VOLUME_NAME}"
VOLUME_PATH = f"/Volumes/{CATALOG}/landing/{VOLUME_NAME}"

print(f"Catálogo objetivo: {CATALOG}")
print(f"Volume objetivo: {VOLUME_PATH}")

Catálogo objetivo: oceanwatch_g06
Volume objetivo: /Volumes/oceanwatch_g06/landing/raw_ais


In [0]:
# Los comentarios documentan propósito y procedencia desde la creación de los activos.
spark.sql(
    f"CREATE CATALOG IF NOT EXISTS {CATALOG} "
    "COMMENT 'Lakehouse OceanWatch Analytics para tráfico marítimo AIS de NOAA.'"
)

for schema_name, comment in SCHEMA_COMMENTS.items():
    spark.sql(
        f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema_name} "
        f"COMMENT '{comment}'"
    )

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {VOLUME_FQN} "
    "COMMENT 'Archivos AIS crudos descargados y CSV descomprimidos; no es una tabla analítica.'"
)

DataFrame[]

## Validación de activos y metadatos

La salida de esta sección deja la evidencia de creación. Si falla, no se continúa con la ingesta.

In [0]:
display(spark.sql(f"SHOW CATALOGS LIKE '{CATALOG}'"))
display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))
display(spark.sql(f"DESCRIBE VOLUME {VOLUME_FQN}"))

catalog
oceanwatch_g06


databaseName
analytics
default
information_schema
landing
reference


name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
raw_ais,oceanwatch_g06,landing,arangurenmarita@gmail.com,,MANAGED,Archivos AIS crudos descargados y CSV descomprimidos; no es una tabla analítica.,VOLUME,VOLUME_DB_STORAGE


In [0]:
# Verificación programática para que una ejecución incompleta no pase inadvertida.
existing_schemas = {next(iter(row.asDict().values())) for row in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()}
missing_schemas = set(SCHEMA_COMMENTS) - existing_schemas
if missing_schemas:
    raise RuntimeError(f"Esquemas no creados: {sorted(missing_schemas)}")

print("CONFIGURATION_OK")
print(f"catalog={CATALOG}")
print(f"volume={VOLUME_PATH}")

CONFIGURATION_OK
catalog=oceanwatch_g06
volume=/Volumes/oceanwatch_g06/landing/raw_ais
